# v3.5 Model – Top150 + FE6 + depth=6

- 목적: v2 baseline 대비, Hybrid Top150 + v3 파생변수 6개 + max_depth=6 세팅으로 성능 개선 여부 확인
- 데이터: df_master_preprocessed_v1.parquet
- 타깃: Segment (0~4, 다중 클래스)
- 사용 알고리즘: XGBoost (multi:softprob, class_weight 적용)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

데이터 / Top150 로드

In [ ]:
df_master = pd.read_parquet("df_master_preprocessed_v1.parquet")
print("Loaded df_master:", df_master.shape)

top150_final = pd.read_parquet("top150_final.parquet")
print("Loaded top150_final:", top150_final.shape)
print(top150_final.head())

TARGET_COL = "Segment"

Train/Val split + class weight

In [ ]:
X_all = df_master.drop(columns=[TARGET_COL])
y_all = df_master[TARGET_COL]

X_train_base, X_val_base, y_train, y_val = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

print("Train (base):", X_train_base.shape, "Val (base):", X_val_base.shape)
print("y_train 분포:")
print(y_train.value_counts())
print("\ny_val 분포:")
print(y_val.value_counts())

classes = np.unique(y_train)
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weights = dict(zip(classes, class_weights_arr))

print("\n[Class Weights]")
for k, v in class_weights.items():
    print(f"Class {k}: {v:.3f}")

sample_weight_train = y_train.map(class_weights)

v3.5 Feature Engineering (Top150 + FE6)

In [ ]:
# 4. v3.5 Feature Engineering ...

base_feats_150 = top150_final["feature"].tolist()
df_fe = df_master.copy()
EPS = 1e-6

# v3 파생변수 6개 생성 ...
# (네 코드 그대로)

feats_v35_custom = [
    "v3_offline_ratio_R3M",
    "v3_big_spend_ratio_R12M",
    "v3_bill_change_R3M_R6M",
    "v3_bill_mean_B5_B2_B0",
    "v3_bill_change_B0_B5",
    "v3_credit_intensity",
]

v35_features = base_feats_150 + feats_v35_custom
v35_features = list(dict.fromkeys(v35_features))

print("\n[v3.5 최종 Feature 개수]:", len(v35_features))
print(v35_features[:30])

In [ ]:
# v3.5 feature 리스트 저장 (이미 너가 만든 버전)
pd.DataFrame({"feature": v35_features}).to_csv(
    "../features/v3_5_feature_list.csv", index=False, encoding="utf-8-sig"
)
with open("../features/v3_5_feature_list.txt", "w", encoding="utf-8") as f:
    for c in v35_features:
        f.write(c + "\n")

X_train/X_val 구성

In [ ]:
X_train_v35 = df_fe.loc[X_train_base.index, v35_features].copy()
X_val_v35   = df_fe.loc[X_val_base.index, v35_features].copy()

print("X_train_v35:", X_train_v35.shape)
print("X_val_v35:", X_val_v35.shape)

In [ ]:
모델 선언/학습/평가

In [ ]:
xgb_v35 = XGBClassifier(
    objective="multi:softprob",
    num_class=len(classes),
    learning_rate=0.05,
    n_estimators=500,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

xgb_v35.fit(
    X_train_v35, y_train,
    sample_weight=sample_weight_train,
    eval_set=[(X_val_v35, y_val)],
    verbose=50
)

y_pred_v35 = xgb_v35.predict(X_val_v35)

macro_f1_v35 = f1_score(y_val, y_pred_v35, average="macro")
print("\n===== 성능 평가 (v3.5: Top150 + FE6 + depth=6) =====")
print("Macro F1 (v3.5):", macro_f1_v35)

print("\nClassification Report (v3.5):")
print(classification_report(y_val, y_pred_v35))

cm_v35 = confusion_matrix(y_val, y_pred_v35)
plt.figure(figsize=(8,6))
sns.heatmap(cm_v35, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (v3.5 - Top150 + FE6 + depth=6)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()